# 03 — Text Analysis & LLM-Based Annotation

This notebook uses the Anthropic API (Claude) to score each post on
linguistic and behavioral dimensions that cannot be captured by simple
statistics. Claude reads each Hebrew post and returns structured scores
that are then merged with the clean dataset for predictive modeling.

**Input:** `data/clean_posts.csv`
**Output:** `data/clean_posts_with_text.csv`

**Sections:**
1. Configuration & Setup
2. Load Data
3. Prompt Design & Single Post Test
4. Score All Posts (with incremental saving)
5. Manual Validation
6. Parse & Merge Results
7. Export

## 1. Configuration & Setup

In [1]:
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import anthropic

# Load API key from .env file
load_dotenv()

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE      = Path("../data/clean_posts.csv")
OUTPUT_FILE     = Path("../data/clean_posts_with_text.csv")
CHECKPOINT_FILE = Path("../data/text_scores_checkpoint.json")

# ── Model ──────────────────────────────────────────────────────────────────
MODEL      = "claude-haiku-4-5"
MAX_TOKENS = 600

# ── API client ─────────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("✓ API client ready")
print(f"  Model:  {MODEL}")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")

✓ API client ready
  Model:  claude-haiku-4-5
  Input:  ../data/clean_posts.csv
  Output: ../data/clean_posts_with_text.csv


## 2. Load Data

In [2]:
df = pd.read_csv(INPUT_FILE)

# Exclude Reels before any scoring
# Reels are video-first with minimal text — linguistic annotation
# is not meaningful for this format. They are kept in clean_posts.csv
# for behavioral EDA (02_eda.ipynb) but excluded here.
reels_excluded = (df['post_type'] == 'Reel').sum()
df = df[df['post_type'] != 'Reel'].copy()

print(f'Loaded {len(df) + reels_excluded} posts total')
print(f'Excluded {reels_excluded} Reels — {len(df)} posts for text analysis')
print()
df[['post_id', 'title', 'date', 'post_type', 'engagement_rate', 'view_through_rate']].head(3)


Loaded 97 posts total
Excluded 7 Reels — 90 posts for text analysis



,post_id,title,date,post_type,engagement_rate,view_through_rate
0,10234810932441995,חופשה עם ילדים זאת מעין חצי חופשה. \n\nלא מקבל...,2025-08-14,Photo,0.056085,0.834921
1,10238066919799644,מלחמה וסטרס עושות משהו מעניין לזמן.\n\nהן מותח...,2026-03-19,Photo,0.105033,0.947484
2,10237996040547707,מה אנחנו באמת רוצים כשאנחנו חושקים במישהו?\n\n...,2026-03-16,Photo,0.011610,0.918539


## 3. Prompt Design & Single Post Test

We first design the scoring prompt and test it on a single post
before running it on the full dataset. This lets us verify the
output format and scoring quality manually.

The prompt instructs Claude to:
- Read a Hebrew Facebook post
- Score it on 15 behavioral and linguistic dimensions
- Return scores as a valid JSON object only — no extra text

In [3]:
SYSTEM_PROMPT = """You are an expert in behavioral psychology and linguistic analysis.
You are analyzing Hebrew Facebook posts written by a single author — a behavioral
neuroscientist and published novelist — who uses social media to share personal
reflections and promote her book.

Your task is to score each post on the dimensions below.
Return ONLY a valid JSON object with no additional text, preamble, or markdown formatting.

Scoring dimensions:

1. emotional_valence: Overall emotional tone of the post
   1=very negative, 2=negative, 3=neutral, 4=positive, 5=very positive

2. emotional_intensity: How emotionally exposed and personally intense IS THE AUTHOR
   HERSELF in this post — not the emotional content of what she is describing, but her
   own emotional state as expressed. A post about a book character feeling something
   scores low even if the described emotion is strong.
   1: low — the author is observational or writing about something external.
      Little personal emotional exposure. Includes posts about others' experiences,
      book characters, or intellectual reflections.
   2: medium — the author expresses some personal feeling but it is moderate,
      present but not overwhelming.
   3: high — the author is raw, vulnerable, or intensely personally affected.
      The post could only have been written from a place of strong personal emotion.
      Examples: missing being pregnant, losing someone close, a deeply personal life moment.

3. tone: Primary tone of the post. Choose one:
   - reflective: personal experience, inner world, first-person emotional writing
   - recommendation: suggesting something to the audience (a book, series, idea, place)
   - book_accomplishment: announcing or celebrating something about the author's own book
   - occasion_anchored: the post is structured around a specific date or external event

4. marketing_mention: Does the post explicitly mention the book is available for purchase,
   name bookstores (סטימצקי, צומת ספרים, עברית), or include a direct call to buy?
   0=no, 1=yes

5. occasion_mention: Does the post reference a specific occasion such as
   ראש השנה, יום האישה, יום הולדת, חגים, מלחמה, or another named event?
   0=no, 1=yes

6. occasion_relevance: If occasion_mention=1, how central is the occasion to the post?
   1=passing mention only, 2=moderate role, 3=the post is primarily about the occasion
   If occasion_mention=0, return 0

7. dominant_pronoun: Which pronoun dominates the overall post body?
   - I: post is primarily first-person (אני, שלי, לי, אותי)
   - You: post primarily addresses the reader (את, אתה, אתם, שלך)
   - We: post uses collective voice (אנחנו, שלנו)
   - Mixed: no single pronoun dominates

8. opening_pronoun: What pronoun appears in the very first sentence?
   Options: I / You / We / None
   None = the first sentence does not clearly use any of these pronouns

9. question_presence: Does the post contain a direct question to the audience?
   0=no, 1=yes

10. list_structure: Is the post structured as a numbered or bullet list,
    or does it use clearly repeated parallel sentence structures?
    0=no, 1=yes

11. narrative_arc: Does the post tell a story with movement or change?
    1=no arc, observation or reflection only
    2=partial arc, some movement but no clear resolution
    3=clear arc with beginning, middle, and end or resolution

12. personal_vulnerability: How much does the author expose something
    genuinely private, risky, or emotionally difficult about herself?

    1 = surface level — the author shares feelings or opinions but nothing
        that feels risky or exposing. Includes emotional posts about external
        topics (the book, other people, events, ideas). The author could have
        written this without feeling personally exposed.

    2 = moderately personal — the author reveals something genuinely private
        but maintains some distance or wraps it in narrative.

    3 = highly vulnerable — the author exposes something intimate, difficult,
        or emotionally raw that most people would hesitate to share publicly.
        The post could only have been written from a place of real courage.

    Key distinction: emotion about external subjects does NOT equal vulnerability.
    A post expressing strong feelings about a book character scores 1.
    A post admitting fear, failure, grief, or a deeply personal life moment scores 3.

13. parenthood_theme: Is parenthood, motherhood, or the author's
    children a meaningful theme in this post?
    0 = not mentioned, or only a passing reference
    1 = children or parenthood play a meaningful role in the post

14. shared_context: Does the post tap into either a major external event
    happening simultaneously, or a significant shared life experience that
    a large portion of the audience has lived through?
    Use the post date provided to judge relevance for external events.

    1 = yes, when the post connects to:
        - A concrete external event happening simultaneously: war, military
          conflict, Jewish holidays, summer vacation season, national news,
          back to school period — even if not explicitly named in the text
        - A major shared life experience tied to a specific life stage:
          pregnancy, new parenthood, loss of a parent, immigration,
          school experiences, marriage, divorce

    0 = no, when:
        - The content is purely personal and idiosyncratic
        - The feeling is universal but not tied to a specific life stage
          or external event (general loneliness, happiness, love)
        - The emotion is relatable but not situationally shared
        - Community or niche events do NOT qualify — book signings,
          literary gatherings, or audience-specific cultural moments
          require broad societal or universal life stage relevance,
          not niche audience affinity

    Key distinction: shared feelings alone do NOT qualify.
    "Missing being pregnant" scores 1 — specific shared life transition.
    "Feeling sad today" scores 0 — universal emotion with no shared context.
    "Attending a book signing" scores 0 — niche community event.
    A post about war anxiety scores 1 even if the word war is not mentioned.

15. tag_count_people: How many individual people are tagged in this post?
    Tagged people appear as proper names (first + last name) WITHOUT @ symbol.
    Look for names at the END in a credits section, mid-post when crediting someone,
    in Latin (e.g. "Kedem Diamant") or Hebrew (e.g. "ליאור שרף") script.
    Be inclusive — if it looks like a person's name being credited, count it.
    Return an integer: 0, 1, 2, 3, etc.

16. explanation: A brief 2-3 sentence explanation of your scores,
    focusing on the dimensions that were most ambiguous or interesting.
    Write in English.

Return your response as a single JSON object with exactly these keys:
emotional_valence, emotional_intensity, tone, marketing_mention,
occasion_mention, occasion_relevance, dominant_pronoun, opening_pronoun,
question_presence, list_structure, narrative_arc, personal_vulnerability,
parenthood_theme, shared_context, tag_count_people, explanation
"""

In [4]:
def score_post(post_text, post_date=None):
    """Send a single post to Claude and return the parsed JSON scores."""
    date_line = f"Post date: {post_date}\n\n" if post_date else ""
    message = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": f"Please score the following Hebrew Facebook post:\n\n{date_line}Post text:\n{post_text}"
            }
        ]
    )

    raw = message.content[0].text.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1]
        raw = raw.rsplit("```", 1)[0].strip()

    return json.loads(raw)

In [85]:
# ── Single post test ────────────────────────────────────────────────────────
# Test on the first post in the dataset
test_post = df.iloc[0]

print("POST TEXT:")
print("─" * 60)
print(test_post["title"])
print("─" * 60)
print(f"Date: {test_post['date']}  |  Type: {test_post['post_type']}")
print(f"Engagement rate: {test_post['engagement_rate']:.3f}  |  VTR: {test_post['view_through_rate']:.3f}")
print()

# Score it
result = score_post(test_post["title"], post_date=test_post["date"])

print("CLAUDE'S SCORES:")
print("─" * 60)
for key, value in result.items():
    if key != "explanation":
        print(f"  {key:<25} {value}")
print()
print("EXPLANATION:")
print(result.get("explanation", "No explanation provided"))

POST TEXT:
────────────────────────────────────────────────────────────
חופשה עם ילדים זאת מעין חצי חופשה. 

לא מקבלים באמת את כל החופש שהיית רוצה ורוב הזמן את די עובדת, אבל כן יש רגעים קסומים. 

כשדניאל קופץ וצוחק בבריכה. כשאני רואה איך לאט לאט נבנה לו הביטחון והוא לא מפחד יותר להיכנס ואפילו רוצה לקפוץ. כשאנחנו מלמדים אותו מזה קייט ופתאום הוא מחפש קייטים בים. 

כשאני מביטה בקפלים המתוקים שברגליים השמנמנות שלו ומנסה לשתות כל רגע כזה לפני שהוא פשוט חולף. 

כי למרות שכל ארוחה במסעדה היא סימפוניה של ״אני רוצה גם״ וכל נסיעה לים יכולה להסתיים בטנטרום כשרוצים כבר ללכת, אני כל הזמן מזכירה לעצמי שזה עוד רגע עובר והם כבר יהיו גדולים ואולי כבר לא יצטרכו אותי כל-כך. 

כי הזמן יש בו משהו מתעתע. הוא נראה כאילו לעולם לא יגמר כשהוא בהתקף טנטרום אבל אז בן רגע הוא פתאם נהיה גדול וכבר לא התינוק הקטן והשמנמן שהחזקתי בידיים לפני רגע. 

אז זאת מעין חצי חופשה כזאת שדורשת יותר עבודה סביב הילד אבל גם ממלאת אותי בו.

בתמונה - כלי הנשק הסודי שלי, גלידה.
──────────────────────────────────────────────────────────

**Manual check:** Read the post above and verify the scores make sense.
If something looks wrong, adjust the prompt in the cell above and re-run the test
before proceeding to score all posts.

## 4. Score All Posts (with incremental saving)

We score all posts one by one, saving results to a checkpoint file after
each post. If the process is interrupted, re-running this cell will skip
already-scored posts and continue from where it left off.

In [5]:
# Load existing checkpoint if available
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    print(f"Loaded checkpoint: {len(checkpoint)} posts already scored")
else:
    checkpoint = {}
    print("No checkpoint found — starting fresh")

# Score all posts
errors = []
total  = len(df)

for _, row in df.iterrows():
    post_id = str(row["post_id"])

    # Skip if already scored
    if post_id in checkpoint:
        continue

    # Skip posts with no text
    if pd.isna(row["title"]) or str(row["title"]).strip() == "":
        errors.append({"post_id": post_id, "error": "empty title"})
        continue

    try:
        scores = score_post(row["title"], post_date=row["date"])
        checkpoint[post_id] = scores

        # Save checkpoint after every post
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(checkpoint, f, ensure_ascii=False, indent=2)

        print(f"✓ [{len(checkpoint)}/{total}] post_id {post_id} — tone: {scores.get('tone')} | valence: {scores.get('emotional_valence')}")

        # Small delay to avoid rate limiting
        time.sleep(0.5)

    except Exception as e:
        print(f"✗ [{len(checkpoint)}/{total}] post_id {post_id} — ERROR: {e}")
        errors.append({"post_id": post_id, "error": str(e)})

print()
print(f"Done. {len(checkpoint)} posts scored, {len(errors)} errors.")
if errors:
    print("Errors:")
    for e in errors:
        print(f"  post_id {e['post_id']}: {e['error']}")

No checkpoint found — starting fresh
✓ [1/90] post_id 10234810932441995 — tone: reflective | valence: 4
✓ [2/90] post_id 10238066919799644 — tone: reflective | valence: 4
✓ [3/90] post_id 10237996040547707 — tone: recommendation | valence: 4
✓ [4/90] post_id 10237942817737170 — tone: book_accomplishment | valence: 4
✓ [5/90] post_id 10237913037112673 — tone: book_accomplishment | valence: 5
✓ [6/90] post_id 10237860987931476 — tone: book_accomplishment | valence: 4
✓ [7/90] post_id 10237836439037769 — tone: reflective | valence: 4
✓ [8/90] post_id 10237816770946079 — tone: reflective | valence: 4
✓ [9/90] post_id 10237734887059033 — tone: reflective | valence: 4
✓ [10/90] post_id 10237705072153679 — tone: book_accomplishment | valence: 4
✓ [11/90] post_id 10237651650578173 — tone: book_accomplishment | valence: 4
✓ [12/90] post_id 10237628211152202 — tone: reflective | valence: 4
✓ [13/90] post_id 10237613761070959 — tone: book_accomplishment | valence: 4
✓ [14/90] post_id 102375890633

### Detect Missing Dimensions

When new scoring dimensions are added after some posts have already been scored,
existing checkpoint entries won't have those dimensions. This cell identifies
which posts are missing any of the expected dimensions and marks them for re-scoring.

Re-running the scoring loop below will then re-score only those posts.

In [6]:
# All dimensions we expect in every scored post
EXPECTED_DIMS = [
    "emotional_valence", "emotional_intensity", "tone",
    "marketing_mention", "occasion_mention", "occasion_relevance",
    "dominant_pronoun", "opening_pronoun", "question_presence",
    "list_structure", "narrative_arc", "personal_vulnerability",
    "parenthood_theme", "shared_context", "tag_count_people", "explanation"
]

# Find posts missing any expected dimension
incomplete = [
    post_id for post_id, scores in checkpoint.items()
    if any(dim not in scores for dim in EXPECTED_DIMS)
]

if incomplete:
    print(f"Found {len(incomplete)} post(s) missing one or more dimensions:")
    for post_id in incomplete:
        missing = [d for d in EXPECTED_DIMS if d not in checkpoint[post_id]]
        print(f"  post_id {post_id} — missing: {missing}")
    print()
    print("Removing incomplete entries from checkpoint so they get re-scored...")
    for post_id in incomplete:
        del checkpoint[post_id]
    print(f"Checkpoint now has {len(checkpoint)} complete entries.")
    print("Re-run the scoring loop below to fill in the missing posts.")
else:
    print(f"All {len(checkpoint)} checkpoint entries have the expected dimensions. Nothing to re-score.")


All 90 checkpoint entries have the expected dimensions. Nothing to re-score.


## 5. Manual Validation

Before merging scores into the dataset, manually inspect a sample of posts
to verify that Claude's scores match your intuition as the author.

This is an important methodological step — it establishes that the annotation
is valid before using it in any predictive model.

In [7]:
# Display a random sample of 5 posts with their scores for manual validation
sample = df.sample(5, random_state=42)

for _, row in sample.iterrows():
    post_id = str(row["post_id"])
    scores  = checkpoint.get(post_id, {})

    print("=" * 65)
    print(f"POST ID: {post_id}  |  Date: {row['date']}  |  Type: {row['post_type']}")
    print(f"Engagement rate: {row['engagement_rate']:.3f}  |  VTR: {row.get('view_through_rate', 'N/A')}  |  Category: {row['post_category']}")
    print()

    # Structural features
    print("STRUCTURAL FEATURES:")
    print(f"  has_hashtag:       {row.get('has_hashtag', 'N/A')}  |  hashtag_count: {row.get('hashtag_count', 'N/A')}")
    print(f"  mentions_children: {row.get('mentions_children', 'N/A')}")
    print()

    # Full post text
    print("FULL TEXT:")
    print(str(row["title"]))
    print()

    # LLM scores
    print("LLM SCORES:")
    for key, value in scores.items():
        if key != "explanation":
            print(f"  {key:<25} {value}")
    print()
    print("EXPLANATION:")
    print(scores.get("explanation", "N/A"))
    print()


POST ID: 10236531633458445  |  Date: 2025-12-07  |  Type: Photo
Engagement rate: 0.085  |  VTR: 0.7819548872180451  |  Category: Viral

STRUCTURAL FEATURES:
  has_hashtag:       1  |  hashtag_count: 1
  mentions_children: 0

FULL TEXT:
#אנשי_הגם_וגם

האלטר אגו שלי הוא שחקנית.

הייתי במגמת תיאטרון בתיכון, יחד עם ביולוגיה (כמובן).

וכשהייתי בתואר השני בירושלים, הלכתי לחוג תיאטרון. פעם בשבוע. 
שנתיים ברצף.

ובסוף כל שנה, העלנו הצגה. איכשהו בכל פעם זו הייתה הצגה של חנוך לוין. 

ואולי זה לא היה במקרה, כי אני אכן מרגישה הזדהות עמוקה עם הדרך שבה הוא מתאר בדידות וכאב ורצון להיות נאהב ולהשתייך ביצירות שלו.ֿ

בהצגה האחרונה שלנו, ״קרום״, שיחקתי את פליציה. דמות שהיא בודדה בפנים ודורכת על אחרים בניסיון שיראו ויאהבו אותה. יש משהו משחרר בלשחק דמויות כאלה קיצוניות. משהו שמאפשר לגעת באזורים בנפש שאסור לגעת בהם ביומיום.

אחד הדוקטורנטים שעבדתי איתו במעבדה, אמר לי שיום אחד לא אוכל לעשות את זה - ללכת לחוג תיאטרון פעם בשבוע ועוד לעלות הצגה בסוף שנה. אצטרך להתמקד, לבחור במשהו אחד, כלומר במחקר. להיות רצינית 

In [89]:
# Look up a specific post by index (change the number to inspect any post)
INSPECT_INDEX = 0  # ← change this to inspect a different post

row     = df.iloc[INSPECT_INDEX]
post_id = str(row["post_id"])
scores  = checkpoint.get(post_id, {})

print("=" * 65)
print(f"POST ID: {post_id}  |  Date: {row['date']}  |  Type: {row['post_type']}")
print(f"Engagement rate: {row['engagement_rate']:.3f}  |  VTR: {row.get('view_through_rate', 'N/A')}  |  Category: {row['post_category']}")
print()

# Structural features
print("STRUCTURAL FEATURES:")
print(f"  has_hashtag:       {row.get('has_hashtag', 'N/A')}  |  hashtag_count: {row.get('hashtag_count', 'N/A')}")
print(f"  has_tag:           {row.get('has_tag', 'N/A')}")
print(f"  mentions_children: {row.get('mentions_children', 'N/A')}")
print()

# Full post text
print("FULL TEXT:")
print(row["title"])
print()

# LLM scores
print("LLM SCORES:")
for key, value in scores.items():
    if key != "explanation":
        print(f"  {key:<25} {value}")
print()
print("EXPLANATION:")
print(scores.get("explanation", "N/A"))


POST ID: 10234810932441995  |  Date: 2025-08-14  |  Type: Photo
Engagement rate: 0.056  |  VTR: 0.834920634920635  |  Category: Algorithm Pushed

STRUCTURAL FEATURES:
  has_hashtag:       0  |  hashtag_count: 0
  has_tag:           N/A
  mentions_children: 1

FULL TEXT:
חופשה עם ילדים זאת מעין חצי חופשה. 

לא מקבלים באמת את כל החופש שהיית רוצה ורוב הזמן את די עובדת, אבל כן יש רגעים קסומים. 

כשדניאל קופץ וצוחק בבריכה. כשאני רואה איך לאט לאט נבנה לו הביטחון והוא לא מפחד יותר להיכנס ואפילו רוצה לקפוץ. כשאנחנו מלמדים אותו מזה קייט ופתאום הוא מחפש קייטים בים. 

כשאני מביטה בקפלים המתוקים שברגליים השמנמנות שלו ומנסה לשתות כל רגע כזה לפני שהוא פשוט חולף. 

כי למרות שכל ארוחה במסעדה היא סימפוניה של ״אני רוצה גם״ וכל נסיעה לים יכולה להסתיים בטנטרום כשרוצים כבר ללכת, אני כל הזמן מזכירה לעצמי שזה עוד רגע עובר והם כבר יהיו גדולים ואולי כבר לא יצטרכו אותי כל-כך. 

כי הזמן יש בו משהו מתעתע. הוא נראה כאילו לעולם לא יגמר כשהוא בהתקף טנטרום אבל אז בן רגע הוא פתאם נהיה גדול וכבר לא התינוק הקטן והשמנמן 

## 6. Parse & Merge Results

Convert the checkpoint dictionary into a dataframe and merge with the
original clean dataset. Each score becomes a new column.

In [8]:
# Convert checkpoint to dataframe
scores_df = pd.DataFrame.from_dict(checkpoint, orient="index")
scores_df.index.name = "post_id"
scores_df = scores_df.reset_index()
scores_df["post_id"] = scores_df["post_id"].astype(df["post_id"].dtype)

print(f"Scores dataframe: {len(scores_df)} rows x {len(scores_df.columns)} columns")
print(f"Score columns: {[c for c in scores_df.columns if c != 'post_id']}")
scores_df.head(3)

Scores dataframe: 90 rows x 17 columns
Score columns: ['emotional_valence', 'emotional_intensity', 'tone', 'marketing_mention', 'occasion_mention', 'occasion_relevance', 'dominant_pronoun', 'opening_pronoun', 'question_presence', 'list_structure', 'narrative_arc', 'personal_vulnerability', 'parenthood_theme', 'shared_context', 'tag_count_people', 'explanation']


,post_id,emotional_valence,emotional_intensity,tone,marketing_mention,occasion_mention,occasion_relevance,dominant_pronoun,opening_pronoun,question_presence,list_structure,narrative_arc,personal_vulnerability,parenthood_theme,shared_context,tag_count_people,explanation
0,10234810932441995,4,3,reflective,0,1,2,I,None,0,0,2,3,1,1,0,This is a deeply vulnerable post where the aut...
1,10238066919799644,4,3,reflective,1,1,2,I,None,1,0,3,3,0,1,2,"This is a deeply personal, reflective post whe..."
2,10237996040547707,4,2,recommendation,0,0,0,I,We,1,0,2,2,0,0,0,This post is primarily a review/recommendation...


In [9]:
# Merge with clean dataset
df_enriched = df.merge(scores_df, on="post_id", how="left")

print(f"Enriched dataset: {len(df_enriched)} rows x {len(df_enriched.columns)} columns")
print(f"Posts with scores: {df_enriched['emotional_valence'].notna().sum()}")
print(f"Posts missing scores: {df_enriched['emotional_valence'].isna().sum()}")

Enriched dataset: 90 rows x 51 columns
Posts with scores: 90
Posts missing scores: 0


In [10]:
# Quick summary of score distributions
score_cols = [
    "emotional_valence", "emotional_intensity", "tone",
    "marketing_mention", "occasion_mention", "occasion_relevance",
    "dominant_pronoun", "opening_pronoun", "question_presence",
    "list_structure", "narrative_arc", "personal_vulnerability",
    "parenthood_theme",
    "shared_context",
    "tag_count_people"
]

print("Score distributions:")
print()
for col in score_cols:
    if col in df_enriched.columns:
        print(f"{col}:")
        print(df_enriched[col].value_counts().sort_index().to_string())
        print()

Score distributions:

emotional_valence:
emotional_valence
3     4
4    60
5    26

emotional_intensity:
emotional_intensity
1     3
2    38
3    49

tone:
tone
book_accomplishment    33
recommendation          9
reflective             48

marketing_mention:
marketing_mention
0    55
1    35

occasion_mention:
occasion_mention
0    65
1    25

occasion_relevance:
occasion_relevance
0    65
1     8
2     9
3     8

dominant_pronoun:
dominant_pronoun
I        74
Mixed     8
We        3
You       5

opening_pronoun:
opening_pronoun
I       25
None    62
We       2
You      1

question_presence:
question_presence
0    56
1    34

list_structure:
list_structure
0    70
1    20

narrative_arc:
narrative_arc
1    33
2    21
3    36

personal_vulnerability:
personal_vulnerability
1    27
2    20
3    43

parenthood_theme:
parenthood_theme
0    54
1    36

shared_context:
shared_context
0    36
1    54

tag_count_people:
tag_count_people
0     27
1     33
2     15
3      8
4      3
6      3
13 

## 7. Export

Save the enriched dataset. This file is the input for `04_modeling.ipynb`.

In [11]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_enriched.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"✓ Exported {len(df_enriched)} posts to {OUTPUT_FILE}")
print(f"  Total columns: {len(df_enriched.columns)}")
print(f"  New text feature columns: {len(score_cols)}")

✓ Exported 90 posts to ../data/clean_posts_with_text.csv
  Total columns: 51
  New text feature columns: 15
